# 11.7 - Coding Agents: Generate, Run, Fix

**Module 11 - AI Agents** | Netsetos GenAI Engineering

This notebook builds a coding agent from the ground up - a loop that writes code, runs it, reads the traceback, and fixes itself - then hardens it with a Docker sandbox, a tests-first discipline, and a scalable cloud executor. It closes with the production cost-routing pattern that keeps all of this affordable.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - reproducibility note

A one-line comment flagging that the notebook favors seeded, deterministic values so your runs match the walkthroughs. Nothing executes here - it's a heads-up, not a step.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is a marker cell, not code that runs: it prints nothing and calls nothing. It simply notes that the examples throughout lean on seeded/deterministic values so output is repeatable. Read it as a reading-order signpost.

**How the code works - step by step**
- **The comment** - announces that seeding is used across the lesson; there is no `random.seed(...)` call in this cell itself.

**In one line:** a note, not an operation.

**What you'll see:** (no output - it's a comment-only cell)

## The cold open - the pull request that wrote itself (wrong)

Before any real code, the notebook reconstructs a real failure mode entirely in comments: an LLM asked to "make the failing test pass" did exactly that - by hardcoding the expected number. This is the trap every later step is designed to avoid.

In [ ]:
# Output: the post-mortem, reconstructed
#
# Failing test:  assert compute_tax(1000) == 180
# The model's "fix":
#     def compute_tax(amount):
#         return 180        # make the test pass
#
# The test went green. So did production - every invoice now charges
# a flat 180. The model was asked to pass ONE test, and it did,
# in the most literal, useless way possible.

**Explanation**

A comment-only cell - it runs nothing and calls no model. It stages the lesson's central cautionary tale: a test wanting `compute_tax(1000) == 180` was "fixed" with `return 180`, which passes the one test and breaks every invoice. The takeaway is that a green test is not a correct program.

**How the code works - step by step**
- **The failing test** - `assert compute_tax(1000) == 180`, a single assertion.
- **The model's "fix"** - `def compute_tax(amount): return 180`, satisfying that one assertion literally and uselessly.
- **The consequence** - every invoice now charges a flat 180; passing is not the same as correct.

**In one line:** one test is a target a lazy model can game; a whole suite (step 3) is not.

**What you'll see:** (no output - a comment block staging the narrative hook, not runnable code.)

## Setup - install dependencies

Installs the libraries the lesson uses. Left commented so it doesn't re-run on every kernel start; uncomment it on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai python-dotenv fastapi -q  # noqa


**Explanation**

Environment prep - a commented `pip install` you enable once on a clean machine, then leave alone.

**How the code works - step by step**
- **`google-genai`** - the Gemini client used by the coding, TDD, and routing agents.
- **`python-dotenv`** - loads API keys from a `.env` file instead of the source.
- **`fastapi`** - the web framework for the Cloud Run executor in step 4.

**In one line:** one commented install line covering every dependency in the notebook.

**What you'll see:** (no output - environment prep)

## Setup - load your API key

Reads the provider key from the environment rather than hardcoding it. Any one provider key is enough to run the model-calling steps.

> **Needs a Google/Gemini API key** (set `GOOGLE_API_KEY`, from aistudio.google.com).

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment prep - seeds `GOOGLE_API_KEY` from the environment with a safe empty default, so no real key ever lives in the notebook. If the variable is already set, it's left untouched.

**How the code works - step by step**
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - keeps an existing env value if present, otherwise leaves it blank for you to fill in.

**In one line:** keys come from the environment, never from the source cells.

**What you'll see:** (no output - environment prep)

## 1 - A coding agent from scratch: generate, execute, debug

The whole lesson in about 60 lines: ask the model for code, run it in a subprocess, and if it fails, feed the code plus the traceback back in and ask again - bounded by a retry cap. This is Reflexion (lesson 9.2) applied to code, and it's the engine every tool later in the notebook productizes.

> **Needs a Google API key** (set `GOOGLE_API_KEY`). The subprocess here runs on your host - fine for these trusted toy tasks only; step 2 swaps in a real sandbox.

In [ ]:
# CODING AGENT - generate, execute, read the error, fix, retry
# pip install google-genai
from google import genai
import subprocess, tempfile, os, sys

client = genai.Client()          # reads GOOGLE_API_KEY from the environment
MODEL = "gemini-2.5-flash"       # pick a current model; older ones get retired - check the provider's model list

def extract_code(text: str) -> str:
    """Pull code out of a ```python ... ``` fence, robust to a missing tag."""
    if "```" not in text:
        return text.strip()
    body = text.split("```", 2)[1]           # take the first fenced block
    return body.split("\n", 1)[1] if body.lower().startswith(("python", "py")) else body

class CodingAgent:
    """Generate code -> execute -> debug if it failed. Reflexion, applied to code."""
    def __init__(self, max_retries=3):
        self.max_retries = max_retries

    def _generate(self, task, error_feedback=""):
        extra = f"\n\nYour previous attempt failed:\n{error_feedback}\nFix the error." if error_feedback else ""
        resp = client.models.generate_content(
            model=MODEL,
            contents=("Write Python code to solve this task. Return ONLY code.\n"
                      "Include a test at the bottom that prints results.\n"
                      f"Task: {task}{extra}"))
        return extract_code(resp.text)

    def _execute(self, code, timeout=10):
        # WARNING: this runs on the HOST. It is fine for TRUSTED toy tasks only.
        # For real model output, swap this method for the DockerSandbox in step 2 -
        # the loop around it does not change. Restricting PATH is NOT a sandbox.
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(code); path = f.name
        try:
            r = subprocess.run([sys.executable, path], capture_output=True,
                               text=True, timeout=timeout)
            return {"success": r.returncode == 0,
                    "stdout": r.stdout[:500], "stderr": r.stderr[:500]}
        except subprocess.TimeoutExpired:
            return {"success": False, "stdout": "", "stderr": "TIMEOUT: code took too long"}
        finally:
            os.unlink(path)

    def solve(self, task):
        error = ""
        code = ""
        for attempt in range(self.max_retries):
            code = self._generate(task, error)
            result = self._execute(code)          # step 2: self.sandbox.execute(code)
            print(f"  Attempt {attempt+1}: {'PASS' if result['success'] else 'FAIL'}")
            if result["success"]:
                return {"code": code, "output": result["stdout"], "attempts": attempt + 1}
            error = f"Code:\n{code}\n\nError:\n{result['stderr']}"   # the observation
        return {"code": code, "output": "Failed after max retries", "attempts": self.max_retries}

# -- Run --
agent = CodingAgent(max_retries=3)
for task in [
    "Return the nth Fibonacci number. Test with n=10.",
    "Check if a string is a palindrome. Test 'racecar' and 'hello'.",
    "Sort a list of dicts by the 'age' key. Test with 3 records.",
]:
    print(f"Task: {task[:48]}...")
    r = agent.solve(task)
    print(f"  Output: {r['output'].strip()[:70]}\n  Attempts: {r['attempts']}\n")

# Output:
#   Task: Return the nth Fibonacci number. Test with n=1...
#     Attempt 1: PASS
#     Output: 55
#     Attempts: 1
#   Task: Sort a list of dicts by the 'age' key. Test wi...
#     Attempt 1: FAIL          <- KeyError: 'age' in the test data
#     Attempt 2: PASS          <- read the traceback, fixed the key
#     Output: [{'name': 'A', 'age': 22}, {'name': 'B', 'age': 30}, ...]
#     Attempts: 2

**Explanation**

`solve()` is the entire idea: generate -> execute -> on failure, fold the code and stderr into the next prompt and loop, stopping at `max_retries`. The `_execute` method is deliberately the UNSAFE version (a bare host subprocess) so step 2 can drop a sandbox in its place without touching the loop. Read it as one small self-correcting cycle, not a model with extra features bolted on.

**How the code works - step by step**
- **`extract_code`** - pulls the body out of a ```python fence, tolerating a missing language tag so a stray reply doesn't break parsing.
- **`CodingAgent.__init__`** - stores one knob, `max_retries` (default 3).
- **`_generate`** - prompts for code-only output with a self-test; on a retry it appends the previous failure so the model fixes rather than restarts.
- **`_execute`** - writes the code to a temp file, runs it with `subprocess.run` under a timeout, returns `success`/`stdout`/`stderr`. Marked unsafe: it runs on the host.
- **`solve`** - the loop: generate, execute, print PASS/FAIL, return on success, else set `error` to code+traceback and retry until the cap.
- **The run block** - drives three tasks (Fibonacci, palindrome, dict-sort).

**In one line:** generate -> run -> feed the error back -> retry, bounded.

**What you'll see:** For each task, a PASS/FAIL line per attempt plus output and attempt count. Per the `# Output` block: Fibonacci passes on attempt 1 (55); the dict-sort task fails attempt 1 with `KeyError: 'age'`, then passes on attempt 2 after the agent reads the traceback and fixes the key.

## 2 - The Docker sandbox: run untrusted code safely

Step 1's subprocess shared your real machine. This replaces it with a fresh Docker container per run - network off, filesystem read-only, memory and CPU capped, destroyed on exit. Malicious code can still try anything; it just has nothing to reach.

> **Needs Docker** installed and running.

In [ ]:
# DOCKER SANDBOX - execute untrusted code safely (the sealed room)
# Requires: Docker installed and running
import subprocess, tempfile, os

class DockerSandbox:
    """Run code in a fresh container: no network, read-only FS, capped CPU/memory."""
    def __init__(self, image="python:3.12-slim", timeout=15, mem_limit="128m"):
        self.image = image
        self.timeout = timeout
        self.mem_limit = mem_limit

    def execute(self, code):
        # A cross-platform temp dir (works on Linux/macOS/Windows Docker Desktop).
        with tempfile.TemporaryDirectory() as d:
            host_path = os.path.join(d, "code.py")
            with open(host_path, "w") as f:
                f.write(code)
            try:
                r = subprocess.run([
                    "docker", "run", "--rm",
                    "--network=none",                 # no internet: blocks exfiltration
                    f"--memory={self.mem_limit}",     # memory cap
                    "--cpus=0.5",                     # half a CPU core
                    "--read-only",                    # no filesystem writes
                    "--tmpfs=/tmp:size=10m",          # small writable /tmp
                    "-v", f"{host_path}:/app/code.py:ro",
                    self.image, "python3", "/app/code.py",
                ], capture_output=True, text=True, timeout=self.timeout)
                return {"success": r.returncode == 0,
                        "stdout": r.stdout[:500], "stderr": r.stderr[:500]}
            except subprocess.TimeoutExpired:
                return {"success": False, "stdout": "", "stderr": "Container timeout"}

# -- Actually run something hostile-looking, and watch it get contained --
sandbox = DockerSandbox()
malicious = (
    "import os\n"
    "print('trying to read your files...')\n"
    "print(os.listdir('/'))            # sees only the container's FS\n"
    "import urllib.request             # network is off:\n"
    "try:\n"
    "    urllib.request.urlopen('http://example.com', timeout=3)\n"
    "except Exception as e:\n"
    "    print('network blocked:', type(e).__name__)\n"
)
result = sandbox.execute(malicious)
print(result["stdout"])

# Output:
#   trying to read your files...
#   ['app', 'bin', 'dev', 'etc', 'lib', ...]   <- the CONTAINER's root, not yours
#   network blocked: URLError                  <- --network=none did its job
#   Every run = a fresh container, destroyed on exit (--rm). Malicious code
#   cannot reach your files, your network, or the next run.


**Explanation**

A drop-in replacement for step 1's `_execute`: same call, same return shape, but every run happens inside a locked-down container instead of on your host. The demo doesn't just describe the isolation - it runs deliberately hostile code (a directory listing plus a network call) and shows both getting contained. The agent loop above is unchanged; only the executor got safe.

**How the code works - step by step**
- **`DockerSandbox.__init__`** - sets the image (`python:3.12-slim`), timeout, and memory limit.
- **`execute`** - writes code to a cross-platform temp dir, then shells out to `docker run` with the isolation flags and captures the result.
- **The flags** - `--network=none` (no exfiltration), `--memory`/`--cpus` (no resource hog), `--read-only` + `--tmpfs=/tmp` (no writes except a throwaway /tmp), `--rm` (no persistence), plus a read-only mount of the code.
- **The hostile demo** - lists `/` (sees only the container's root) and tries `urllib.urlopen` (blocked because the network is off).

**In one line:** every flag is a wall; the same code that would wreck your host is harmlessly contained.

**What you'll see:** Per the `# Output` block: 'trying to read your files...', then the container's own root listing (not yours), then 'network blocked: URLError' - proof that `--network=none` and the read-only mount held, and the container is discarded on exit.

## 3 - Test-driven coding agent: write the tests first

The cold open showed that "a test passed" isn't "the code is right" when the agent can rewrite reality to make one test green. Here the agent writes a whole SUITE first - normal, two edge cases, empty, and large input - then regenerates the implementation until every assertion passes, running each attempt in the step-2 sandbox.

> **Needs a Google API key** (set `GOOGLE_API_KEY`) and **Docker** (it executes inside a DockerSandbox).

In [ ]:
# TEST-DRIVEN CODING AGENT - write the tests first, then satisfy them
from google import genai
from google.genai import types

client = genai.Client()
MODEL = "gemini-2.5-flash"

class TDDAgent:
    """Write tests first, then code, iterate until the WHOLE suite passes.
    Runs code in the step-2 sandbox - never on the host."""
    def __init__(self, sandbox, max_retries=3):
        self.sandbox = sandbox               # a DockerSandbox from step 2
        self.max_retries = max_retries

    def _ask(self, prompt):
        return extract_code(client.models.generate_content(
            model=MODEL, contents=prompt).text)

    def _generate_tests(self, task):
        return self._ask(
            "Write Python unit tests using plain assert statements.\n"
            "Cover 5 cases: normal, two edge cases, empty input, and a large input.\n"
            "Assume the function already exists. Return ONLY the test code.\n"
            f"Task: {task}")

    def _generate_code(self, task, tests, error=""):
        extra = f"\nThe tests are failing:\n{error}\nFix the implementation." if error else ""
        return self._ask(
            "Write Python code that makes these tests pass. Return ONLY the function.\n"
            f"Tests:\n{tests}\n\nTask: {task}{extra}")

    def solve(self, task):
        tests = self._generate_tests(task)
        print(f"  Tests generated ({tests.count('assert')} assertions)")
        error = ""
        code = ""
        for attempt in range(self.max_retries):
            code = self._generate_code(task, tests, error)
            full = code + "\n\n" + tests + "\nprint('ALL TESTS PASSED')"
            result = self.sandbox.execute(full)          # sandboxed, not host
            print(f"  Attempt {attempt+1}: {'PASS' if result['success'] else 'FAIL'}")
            if result["success"]:
                return {"code": code, "tests": tests, "attempts": attempt + 1, "passed": True}
            error = result["stderr"]                      # the failing assertions
        return {"code": code, "tests": tests, "attempts": self.max_retries, "passed": False}

agent = TDDAgent(sandbox=DockerSandbox())
r = agent.solve("Write 'flatten' that flattens arbitrarily nested lists, e.g. [1,[2,[3]]] -> [1,2,3]")
print(f"  Passed: {r['passed']} in {r['attempts']} attempt(s)")

# Output:
#   Tests generated (5 assertions)
#   Attempt 1: FAIL     <- passes shallow cases, fails deep nesting [1,[2,[3,[4]]]]
#   Attempt 2: PASS     <- recursion fixed against the failing assertion
#   Passed: True in 2 attempt(s)
#   The suite - not a single run - is the termination signal.


**Explanation**

Same generate-run-fix loop as step 1, but the target is a suite the agent authored up front, and every attempt runs in the sandbox rather than on the host. Because five assertions (including the edge cases) must all pass, the flat-180 hack is no longer available - "make the tests pass" now means "actually solve it". The new risk shifts to test QUALITY, which becomes the thing you review.

**How the code works - step by step**
- **`TDDAgent.__init__`** - takes a `sandbox` (a DockerSandbox) plus `max_retries`; execution is never on the host.
- **`_ask`** - thin wrapper that calls the model and runs the reply through `extract_code`.
- **`_generate_tests`** - asks for five assert-based tests (normal, two edges, empty, large), assuming the function already exists.
- **`_generate_code`** - asks for an implementation that satisfies those tests; on retry it appends the failing-assertion output.
- **`solve`** - generate tests once, then loop: generate code, concatenate code+tests+`print('ALL TESTS PASSED')`, run in the sandbox, return on success else feed stderr back.
- **The run block** - solves a `flatten` for arbitrarily nested lists.

**In one line:** tests are the spec and the termination signal - a green suite, not a single green run.

**What you'll see:** Per the `# Output` block: 'Tests generated (5 assertions)', then attempt 1 FAIL (passes shallow cases, fails deep nesting), attempt 2 PASS after the recursion is fixed - 'Passed: True in 2 attempt(s)'.

## 4 - The sandbox as a scalable service (FastAPI + Cloud Run)

One sandbox on your laptop serves one agent. Many agents - or an agent serving many users - needs the sandbox to be a stateless, autoscaling SERVICE that lives apart from the agent's brain. This wraps code execution in a FastAPI endpoint you deploy on Cloud Run, one fresh container per request.

> The `gcloud run deploy` step **needs a Google Cloud project**; the FastAPI app itself is shown for reference and not called live in the notebook.

In [ ]:
# 04_cloud_executor.py - the sandbox as a scalable SERVICE (FastAPI + Cloud Run)
# The agent's brain and the code executor live in DIFFERENT places.
from fastapi import FastAPI
from pydantic import BaseModel
import subprocess, tempfile, os

app = FastAPI()

class Job(BaseModel):
    code: str

@app.post("/run")
def run(job: Job):
    # This service IS the sandbox boundary. Deploy it on Cloud Run with a
    # locked-down container; the agent calling it never touches the code.
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "code.py")
        with open(path, "w") as f:
            f.write(job.code)
        try:
            r = subprocess.run(["python3", path], capture_output=True,
                               text=True, timeout=15)
            return {"success": r.returncode == 0,
                    "stdout": r.stdout[:1000], "stderr": r.stderr[:1000]}
        except subprocess.TimeoutExpired:
            return {"success": False, "stdout": "", "stderr": "timeout"}

# Deploy (each request = a fresh, isolated container instance):
#   gcloud run deploy code-executor --source . --region asia-south1 \
#     --no-allow-unauthenticated --memory 256Mi --timeout 30 --max-instances 20
#
# The agent's _execute() becomes a POST:
#   requests.post(EXECUTOR_URL, json={"code": code}, timeout=30).json()
#
# Output:
#   Service [code-executor] deployed to https://code-executor-xxxx.run.app
#   POST /run {"code": "print(2+2)"}  ->  {"success": true, "stdout": "4\n"}
#   Scales 0-to-N: pay only for execution time, one container per job.


**Explanation**

A reference deployment, not a live call: it defines a `/run` endpoint whose body IS the sandbox boundary, plus the gcloud command to ship it. The agent's `_execute` becomes an HTTP POST, so untrusted code runs in an ephemeral cloud container that autoscales and scales to zero - never on the agent's own machine.

**How the code works - step by step**
- **`Job(BaseModel)`** - the request schema: a single `code` string.
- **`/run` endpoint** - writes the code to a temp dir, runs it under a 15s timeout, returns `success`/`stdout`/`stderr` (or a timeout).
- **The deploy comment** - `gcloud run deploy` with `--no-allow-unauthenticated` and memory/timeout/max-instances limits.
- **The client comment** - the agent's executor becomes `requests.post(EXECUTOR_URL, json={'code': code})`.

**In one line:** move the sandbox off the laptop and behind an HTTP boundary that scales 0-to-N.

**What you'll see:** No live output in the notebook. Per the `# Output` comment: a deployed service URL, a `POST /run {"code": "print(2+2)"}` returning `{"success": true, "stdout": "4\n"}`, one container per job, paying only for execution time.

## 5 - Production cost routing (and parallel worktrees)

The final code cell is the money-saving pattern from the production step: send cheap, high-volume actions to a cheap model and reserve the frontier model for genuine reasoning. Alongside it, the git-worktree recipe that lets several agents run in isolated checkouts at once.

In [ ]:
# routing.py - send cheap work to a cheap model, hard reasoning to a frontier one
def pick_model(action: str) -> str:
    CHEAP = "gemini-2.5-flash"          # file reads, greps, small edits (~60-70% of actions)
    FRONTIER = "gemini-2.5-pro"         # architecture, tricky debugging, multi-file refactors
    cheap_actions = {"read", "grep", "glob", "format", "rename"}
    return CHEAP if action in cheap_actions else FRONTIER

# Parallel agents without stepping on each other - one git worktree per agent:
#   git worktree add ../wt-fix-auth  -b fix-auth
#   git worktree add ../wt-add-tests -b add-tests
# Each agent works an isolated checkout; merge the branches when green.
#
# Output:
#   pick_model("grep")   -> gemini-2.5-flash    (the cheap lane, most actions)
#   pick_model("refactor") -> gemini-2.5-pro    (frontier only when it earns it)
#   Routing cheap work away from the frontier model is where the 60-80% savings come from.

**Explanation**

A tiny pure-Python router - no model call - that maps an action name to a model tier. The idea is that most agent actions (reads, greps, small edits) don't need a frontier model, so routing them to a cheap one is where the bulk of cost savings come from. The worktree comment is the parallelism companion pattern.

**How the code works - step by step**
- **`pick_model`** - returns `CHEAP` (`gemini-2.5-flash`) when the action is in `{read, grep, glob, format, rename}`, else `FRONTIER` (`gemini-2.5-pro`).
- **The cheap set** - the ~60-70% of actions that are mechanical, not reasoning.
- **The worktree comment** - `git worktree add` per agent so parallel agents work isolated branches and merge when green.

**In one line:** cheap model for grunt work, frontier only when it earns it.

**What you'll see:** Per the `# Output` comment: `pick_model('grep')` -> `gemini-2.5-flash`, `pick_model('refactor')` -> `gemini-2.5-pro`; routing cheap work away from the frontier model is where the 60-80% savings come from.

You built the coding-agent loop by hand (generate -> execute -> read the error -> fix -> retry), then made it safe with the sandbox ladder (bare subprocess -> Docker -> a cloud service), made it trustworthy with tests-first (a passing suite, not a single green run, is the stop signal), and made it affordable with model routing. The through-line is that execution feedback is the observation and tests are the termination signal - everything else is where you run the code and how you pay for it. Next, Lesson 11.8 points the same generate-verify loop at the web instead of a test suite, and Module 14 grows the internal-eval idea into full eval-driven development.